<a href="https://colab.research.google.com/github/rajarshee99/RAG-/blob/main/RAG_with_MRL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from openai import OpenAI

# ✅ Set env variable
os.environ["OPENROUTER_API_KEY"] = "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"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

In [2]:
!pip install sentence_transformers
!pip install faiss-cpu
!pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 53.0 MB/s eta 0:00:00


In [5]:
import faiss
import numpy as np
import requests
import os

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "YOUR_OPENROUTER_API_KEY")
EMBED_MODEL        = "nvidia/llama-nemotron-embed-vl-1b-v2:free"
CHAT_MODEL         = "openai/gpt-4o-mini" # Changed from "qwen/qwen3-235b-a22b:free"

# MRL: full dim for Nemotron-Embed-1B is 4096
# Matryoshka lets you truncate to any prefix and still get meaningful embeddings.
# Smaller dim = faster search, less memory, slight accuracy tradeoff.
# Common MRL dims: 4096 (full), 2048, 1024, 512, 256, 128
MRL_DIM = 1024  # <-- tune this: 2048 for higher accuracy, 256 for speed

# ---------------------------------------------------------------
# TRACE LOGGER
# ---------------------------------------------------------------
trace = []

def log(stage, data):
    print(f"\n{'='*60}")
    print(f"  STAGE: {stage}")
    print(f"{'='*60}")
    print(data)
    trace.append({"stage": stage, "data": str(data)})


# ---------------------------------------------------------------
# EMBEDDING FUNCTION  (OpenRouter → Nemotron)
# ---------------------------------------------------------------
def get_embeddings(texts: list[str], mrl_dim: int = MRL_DIM) -> np.ndarray:
    """
    Calls OpenRouter's embedding endpoint for the Nemotron model.
    MRL truncation: slice the first `mrl_dim` dimensions, then L2-normalize.
    L2-normalization is REQUIRED for cosine similarity via IndexFlatIP.
    """
    url = "https://openrouter.ai/api/v1/embeddings"
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": EMBED_MODEL,
        "input": texts,
    }

    resp = requests.post(url, headers=headers, json=payload)
    resp.raise_for_status()
    data = resp.json()

    # Parse embeddings in order
    embeddings = [item["embedding"] for item in sorted(data["data"], key=lambda x: x["index"])]
    vectors = np.array(embeddings, dtype=np.float32)

    # ── MRL TRUNCATION ───────────────────────────────────────
    # Matryoshka models are trained so that any prefix of the full
    # embedding vector is itself a valid, useful embedding.
    # Truncating to `mrl_dim` gives you a smaller representation
    # that retains most of the semantic information.
    full_dim = vectors.shape[1]
    if mrl_dim > full_dim:
        raise ValueError(f"MRL dim {mrl_dim} exceeds model output dim {full_dim}")
    vectors = vectors[:, :mrl_dim]   # <── THE MRL STEP
    # ─────────────────────────────────────────

    # ── L2 NORMALIZE → enables cosine sim via dot product ─────
    # After L2-norm: cos(a,b) = dot(a,b)  because |a|=|b|=1
    # FAISS IndexFlatIP then performs exact cosine search.
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1e-10, norms)  # avoid div-by-zero
    vectors = vectors / norms
    # ─────────────────────────────────────────

    return vectors


# ---------------------------------------------------------------
# KNOWLEDGE BASE
# ---------------------------------------------------------------
documents = [
"""
# Retrieval-Augmented Generation (RAG)

## Overview
RAG (Retrieval-Augmented Generation) combines retrieval with generation.
It allows Large Language Models (LLMs) to access external knowledge at inference time
instead of relying only on their training data.

LLMs have:
- Static knowledge (frozen at training cutoff)
- No access to private or real-time data
- A tendency to hallucinate when unsure

RAG solves this by retrieving relevant information and injecting it into the prompt before generating a response.

## Core Idea
Instead of relying purely on model weights, RAG enables the model to:
1. Retrieve relevant context from an external source
2. Use that context to generate grounded answers

## Key Components

### 1. Retriever
- Finds relevant information from a knowledge base
- Uses semantic search (embeddings) instead of keyword matching
- Example:
  Query: "Why did the server crash?"
  Retrieved: "The process was OOM-killed"

### 2. Generator (LLM)
- Input: System prompt + Retrieved context + User query
- Output: Answer grounded in retrieved data

## Embeddings and Vector Databases
- Embeddings convert text into vectors
- Vector databases store and search these vectors
- Enable semantic similarity search

## Traditional RAG Workflow
1. Chunk documents
2. Convert chunks into embeddings
3. Store in vector database
4. At query time:
   - Embed user query
   - Retrieve top-k similar chunks
   - Pass them to LLM
   - Generate answer

## Vectorless RAG
Vectorless RAG avoids embeddings and uses reasoning over document structure.

### When It Works Best
- Single structured document
- Clear hierarchy (like PDFs, manuals, reports)

### When It Struggles
- Multiple documents
- Noisy or irrelevant data

## Vector vs Vectorless RAG

| Aspect            | Vector RAG           | Vectorless RAG        |
|-------------------|----------------------|-----------------------|
| Retrieval method  | Embedding similarity | Logical reasoning     |
| Best for          | Large, multi-doc     | Single structured doc |
| Scalability       | High                 | Limited               |
| Noise handling    | Strong               | Weak                  |

## Vectorless RAG Workflow (PageIndex)
1. Parse document structure → build hierarchical ToC tree
2. LLM reasons over tree → picks relevant section
3. Navigate → retrieve section text
4. Generate answer grounded in that section

## Key Insight
Traditional RAG: "Find similar meaning using vectors"
Vectorless RAG:  "Reason about where the answer should be"
"""
]


# ---------------------------------------------------------------
# QUERY
# ---------------------------------------------------------------
query = (
    "What's the difference between Traditional RAG vs Vectorless RAG? "
    "Give me a structured answer. After the answer show me the reasoning "
    "steps you have done in ASCII tree graph style."
)
log("QUERY", query)


# ---------------------------------------------------------------
# STEP 1 — EMBED DOCUMENTS  (MRL + cosine-ready)
# ---------------------------------------------------------------
log("EMBEDDING DOCUMENTS", f"Model: {EMBED_MODEL}  |  MRL dim: {MRL_DIM}")
doc_embeddings = get_embeddings(documents, mrl_dim=MRL_DIM)
log("DOC EMBEDDINGS SHAPE", doc_embeddings.shape)
log("DOC EMBEDDING SAMPLE (first 10 dims)", doc_embeddings[0][:10])


# ---------------------------------------------------------------
# STEP 2 — BUILD FAISS INDEX  (IndexFlatIP = cosine sim)
# ---------------------------------------------------------------
# Why IndexFlatIP instead of IndexFlatL2?
#   L2 distance = ||a - b||²  = 2 - 2·cos(a,b)  when vectors are normalized
#   Inner product on L2-normalized vectors = cos(a,b) directly
#   IndexFlatIP returns HIGHER score = MORE similar  (vs L2: lower = better)
index = faiss.IndexFlatIP(MRL_DIM)
index.add(doc_embeddings)
log("FAISS INDEX", f"Type: IndexFlatIP | Vectors stored: {index.ntotal} | Dim: {MRL_DIM}")


# ---------------------------------------------------------------
# STEP 3 — EMBED QUERY  (same MRL dim, same L2-norm)
# ---------------------------------------------------------------
log("EMBEDDING QUERY", query[:80] + "...")
q_embedding = get_embeddings([query], mrl_dim=MRL_DIM)
log("QUERY EMBEDDING SAMPLE (first 10 dims)", q_embedding[0][:10])


# ---------------------------------------------------------------
# STEP 4 — VECTOR SEARCH  (cosine similarity, top-k=3)
# ---------------------------------------------------------------
TOP_K = 3
scores, indices = index.search(q_embedding, TOP_K)
log("COSINE SIMILARITY SCORES", scores.tolist())   # higher = more similar
log("RETRIEVED INDICES", indices.tolist())


# ---------------------------------------------------------------
# STEP 5 — RETRIEVE CHUNKS
# ---------------------------------------------------------------
retrieved_chunks = [documents[i] for i in indices[0] if i < len(documents)]
log("RETRIEVED CHUNKS COUNT", len(retrieved_chunks))
for idx, chunk in enumerate(retrieved_chunks):
    log(f"CHUNK {idx+1} (first 200 chars)", chunk[:200])


# ---------------------------------------------------------------
# STEP 6 — BUILD PROMPT
# ---------------------------------------------------------------
context = "\n\n---\n\n".join(retrieved_chunks)
final_prompt = f"""Answer ONLY from the context below. Do not use prior knowledge.

Context:
{context}

Question:
{query}
"""
log("FINAL PROMPT (first 300 chars)", final_prompt[:300])


# ---------------------------------------------------------------
# STEP 7 — GENERATE ANSWER  (OpenRouter chat)
# ---------------------------------------------------------------
chat_url = "https://openrouter.ai/api/v1/chat/completions"
chat_payload = {
    "model": CHAT_MODEL,
    "messages": [
        {
            "role": "system",
            "content": (
                "You are a senior AI/ML engineer with deep expertise in RAG systems. "
                "Answer with high technical precision. Be structured and concise."
            ),
        },
        {"role": "user", "content": final_prompt},
    ],
    "temperature": 0.3,
}

chat_resp = requests.post(
    chat_url,
    headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}", "Content-Type": "application/json"},
    json=chat_payload,
)
chat_resp.raise_for_status()
answer = chat_resp.json()["choices"][0]["message"]["content"]

log("LLM ANSWER", answer)
print("\n\n" + "="*60)
print("FINAL ANSWER")
print("="*60)
print(answer)


# ---------------------------------------------------------------
# STEP 8 — FULL TRACE SUMMARY
# ---------------------------------------------------------------
print("\n\n" + "="*60)
print("PIPELINE TRACE SUMMARY")
print("="*60)
for step in trace:
    print(f"  [{step['stage']}]")


  STAGE: QUERY
What's the difference between Traditional RAG vs Vectorless RAG? Give me a structured answer. After the answer show me the reasoning steps you have done in ASCII tree graph style.

  STAGE: EMBEDDING DOCUMENTS
Model: nvidia/llama-nemotron-embed-vl-1b-v2:free  |  MRL dim: 1024

  STAGE: DOC EMBEDDINGS SHAPE
(1, 1024)

  STAGE: DOC EMBEDDING SAMPLE (first 10 dims)
[ 0.00156661 -0.00624479 -0.00582775  0.04012267  0.033515    0.05195149
 -0.03342834  0.03141354 -0.01348616  0.02920376]

  STAGE: FAISS INDEX
Type: IndexFlatIP | Vectors stored: 1 | Dim: 1024

  STAGE: EMBEDDING QUERY
What's the difference between Traditional RAG vs Vectorless RAG? Give me a struc...

  STAGE: QUERY EMBEDDING SAMPLE (first 10 dims)
[-0.02332177  0.00060158  0.00304798 -0.02196097  0.00277969 -0.01293319
 -0.00614022 -0.00641128 -0.02836671  0.07058488]

  STAGE: COSINE SIMILARITY SCORES
[[0.3869522213935852, -3.4028234663852886e+38, -3.4028234663852886e+38]]

  STAGE: RETRIEVED INDICES
[[0, -